In [1]:
%pylab
%matplotlib inline

Using matplotlib backend: <object object at 0x10a8b2650>
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [1]:
import os
import gc
import sys
import matplotlib.pyplot as plt
 #from scipy.ndimage import gaussian_filter
if sys.platform!='darwin':
    from cupyx.scipy.ndimage import gaussian_filter
    import cupy as cp
else:
    from scipy.ndimage import gaussian_filter
    import pyclesperanto_prototype as cle
    # initialize GPU
    device = cle.select_device("GTX")
    print("Used GPU: ", device)
 

import numpy as np
from os.path import getsize
import napari
from skimage.io import imread

filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'
MAXCHUNKSIZE = 1024*288*2*1500 
class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)
        self.app = napari.Viewer()
        

        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)

        l = self.app.layers['Image']
        
        l.data = self.array
        
    def getImage(self,n):

        offset = n*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize)
        nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,self.height,self.width))
        
        return nparray

    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = nFrames
        totalFrames = end-start
        totalFramesSize = totalFrames*self.frameSize

        if totalFramesSize<=MAXCHUNKSIZE:
            #stack = np.zeros((self.height,self.width,totalSize),dtype=np.uint16)
            #for i in range(start,end,step):
            #    stack[:,:,i-start] = gaussian_filter(self.getImage(i),2)
            #    #stack.append(getImage(r,i,width,height))

            offset = start*self.frameSize
            self.r.seek(offset)
            st = self.r.read(totalFramesSize)
            if sys.platform != 'darwin':
                stack = cp.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))
                stack = gaussian_filter(stack,(2,2,2))
                stack3 = [cp.asnumpy(stack)]
            else:
                stack = np.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))

                stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                stack3 = [stack]


            
        else:
            chunksizeFrames = MAXCHUNKSIZE//(self.frameSize)  #number of frames in a chunk
            nchunks = totalFrames//chunksizeFrames
            remainderFrames = totalFrames%chunksizeFrames
            stack3 = []
            for i in range(nchunks):
                offset = (start+i*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*chunksizeFrames)
                if sys.platform !='darwin':
                    stack = cp.frombuffer(st,dtype = np.uint16).reshape((chunksizeFrames,self.height,self.width))

                    stack = gaussian_filter(stack,(2,2,2))
                    stack3.append(cp.asnumpy(stack))
                else:
                    stack = np.frombuffer(st,dtype = np.uint16).reshape((chunksizeFrames,self.height,self.width))
                    stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                    stack3.append(stack)

            if remainderFrames != 0:
                offset = (start+nchunks*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*remainderFrames)
                if sys.platform != 'darwin':
                    stack = cp.frombuffer(st,dtype = np.uint16).reshape((remainderFrames,self.height,self.width))

                    stack = gaussian_filter(stack,(2,2,2))
                    stack3.append(cp.asnumpy(stack))
                else:
                    stack = np.frombuffer(st,dtype = np.uint16).reshape((remainderFrames,self.height,self.width))
                    stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                    stack3.append(stack) 
        if sys.platform !='darwin':         
            cp._default_memory_pool.free_all_blocks()    
        return stack3

    def loadNextNFrames(self,n):
        newStacks = self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)
        self.array= np.vstack([self.array, *newStacks])
        del newStacks
        try:
            l = self.app.layers['Image']
            l.data = self.array
        except:
            self.app.add_image(self.array)
            self.app.add_shapes(None, shape_type='rectangle', name='Shapes',  edge_width=3, face_color=np.array([0,0,0,0]),edge_color = 'red',edge_color_cycle=plt.rcParams['axes.prop_cycle'].by_key()['color'])


        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return
        newStack = self.loadWholeStack(start=self.currentLastFrame,end = n)
        self.array= np.vstack([self.array, *newStack])

        if self.currentLastFrame == 0:
                self.app.add_image(self.array)
                self.app.add_shapes(None, shape_type='rectangle', name='Shapes',  edge_width=3, face_color=np.array([0,0,0,0]),edge_color = 'red',edge_color_cycle=plt.rcParams['axes.prop_cycle'].by_key()['color'])
        else:
            l = self.app.layers['Image']
            l.data = self.array
        self.currentLastFrame = self.array.shape[0]



Used GPU:  <Apple M1 Pro on Platform: Apple (2 refs)>


In [2]:
#cp._default_memory_pool.free_all_blocks()
fn = '../sampleImage/'
try:
    tb.r.close()
    del tb.array
    tb.loadFile(fn)
    gc.collect()
except:
    print('nope')
    tb = thorlabsFile(fn)

nope


In [10]:
tb.loadUpToFrameN(200)

In [13]:
import ipywidgets as widgets
from ipywidgets import interact, Dropdown
import plotly.graph_objs as go

def jupyterPy(tb):
    plotButton =  widgets.Button(description='Plot ROIs',button_style = 'primary')
    loadNextButton =  widgets.Button(description='Load Next 500',button_style = 'primary')

    fig = go.FigureWidget(data=[],layout=go.Layout({ 'autosize':True,'height':750, }))
    ui = widgets.VBox([plotButton,loadNextButton,fig])
    l = tb.app.layers['Shapes']

    def on_plot_clicked(change):
        fig.data = []
        for el in l.data:
            xmin,xmax = int(el[:,0].min()),int(el[:,0].max())
            ymin,ymax = int(el[:,1].min()),int(el[:,1].max())
            z = tb.array[:,xmin:xmax,ymin:ymax].mean(1).mean(1)
            fig.add_scatter(y=z)
    
    def on_loadNext_clicked(change):
        tb.loadNextNFrames(500)

    plotButton.on_click(on_plot_clicked)
    loadNextButton.on_click(on_loadNext_clicked)
    display(ui)

In [14]:
jupyterPy(tb)

In [5]:
for el in l.data:
    xmin,xmax = int(el[:,0].min()),int(el[:,0].max())
    ymin,ymax = int(el[:,1].min()),int(el[:,1].max())
    z = tb.array[:,xmin:xmax,ymin:ymax].mean(1).mean(1)
    plt.plot(z)
plt.show()

NameError: name 'l' is not defined

# Use this to load N extra frames. Don't load too many frames (around 500) at once (yet)

['#1f77b4',
 '#ff7f0e',
 '#2ca02c',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2',
 '#7f7f7f',
 '#bcbd22',
 '#17becf']

In [4]:
tb.loadNextNFrames(500)

# Use this to load up to frame N

In [3]:
tb.loadUpToFrameN(100)

In [7]:
tb.nFrames

7330

# Use this to load another file

In [9]:
tb.loadFile('E:\\Today\\2023_01_30_Atoh1-Cre-GCaMP6s_P4_in vivo\\1_003') 

In [3]:
import cupy as cp
import numpy as np
import dask.array as da

huge_array = da.ones(
    (5000, 500, 200), 
    chunks=(5000, 5000, 5), 
    dtype=float)

huge_array.nbytes / 1e9 # 40 GB in size

np.prod(huge_array.chunksize, dtype=float) * huge_array.dtype.itemsize / 1e9 # chunk size of 1 GB

huge_array = huge_array.map_blocks(cp.asarray) # make it a CuPy-backed array

array_sum = da.sum(huge_array)
array_sum.compute()

array(5.e+08)

In [7]:
import numpy as np
from skimage import data
import napari


blobs = data.binary_blobs(
    length=128, blob_size_fraction=0.05, n_dim=3, volume_fraction=0.1
).astype(float)

viewer = napari.view_image(blobs.astype(float))

# create one random polygon per "plane"
planes = np.tile(np.arange(128).reshape((128, 1, 1)), (1, 5, 1))
np.random.seed(0)
corners = np.random.uniform(0, 128, size=(128, 5, 2))
shapes = np.concatenate((planes, corners), axis=2)

base_cols = ['red', 'green', 'blue', 'white', 'yellow', 'magenta', 'cyan']
colors = np.random.choice(base_cols, size=128)

layer = viewer.add_shapes(
    np.array(shapes),
    shape_type='polygon',
    face_color=colors,
    name='sliced',
)

masks = layer.to_masks(mask_shape=(128, 128, 128))
labels = layer.to_labels(labels_shape=(128, 128, 128))
shape_array = np.array(layer.data)

print(
    f'sliced: nshapes {layer.nshapes}, mask shape {masks.shape}, '
    f'labels_shape {labels.shape}, array_shape, {shape_array.shape}'
)

corners = np.random.uniform(0, 128, size=(2, 2))
layer = viewer.add_shapes(corners, shape_type='rectangle', name='broadcasted')

masks = layer.to_masks(mask_shape=(128, 128))
labels = layer.to_labels(labels_shape=(128, 128))
shape_array = np.array(layer.data)

print(
    f'broadcast: nshapes {layer.nshapes}, mask shape {masks.shape}, '
    f'labels_shape {labels.shape}, array_shape, {shape_array.shape}'
)

if __name__ == '__main__':
    napari.run()

sliced: nshapes 128, mask shape (128, 128, 128, 128), labels_shape (128, 128, 128), array_shape, (128, 5, 3)
broadcast: nshapes 1, mask shape (1, 128, 128), labels_shape (128, 128), array_shape, (1, 4, 2)
